# Ансамбли

Пункты чеклиста: усреднение, voting и stacking через линейную регрессию и Ridge. Базовые модели берём с лучшими гиперпараметрами из прошлых ноутбуков и сначала оцениваем каждую по отдельности через ту же схему валидации, чтобы ансамбли было с чем сравнивать

Схема валидации как в `dnn.ipynb`: KFold с 5 фолдами и препроцессингом внутри каждого фолда. Признаки, предобработка и таргет те же, что в прошлых ноутбуках, модули с общим кодом ещё не вынесены, поэтому в разделе 1 всё повторяется заново

## 1. Признаки из прошлых ноутбуков

Все шаги, принятые в `baseline_and_preprocessing.ipynb` и `feature_engineering.ipynb`

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from xgboost import XGBRegressor

pd.set_option("display.max_columns", 100)

DATA_DIR = Path("../data")
TARGET = "SalePrice"
RANDOM_STATE = 42

train = pd.read_csv(DATA_DIR / "train.csv")
X = train.drop(columns=[TARGET, "Id"])
y = train[TARGET]

In [2]:
GARAGE_COLUMNS = ["GarageType", "GarageFinish", "GarageQual", "GarageCond"]
BASEMENT_COLUMNS = ["BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"]
NO_OBJECT_COLUMNS = ["Alley", "Fence", "PoolQC", "FireplaceQu", "MiscFeature"] + GARAGE_COLUMNS + BASEMENT_COLUMNS
NO_ZERO_AREA_COLUMNS = ["LotFrontage", "LotArea", "1stFlrSF", "GrLivArea"]
DROPPED_COLUMNS = [
    "GarageArea", "GarageYrBlt", "TotRmsAbvGrd", "1stFlrSF",
    "Utilities", "Street", "Condition2", "RoofMatl", "Heating",
]
ORDINAL_QUALITY_COLUMNS = ["ExterQual", "BsmtQual", "BsmtCond", "HeatingQC", "KitchenQual", "FireplaceQu"]
QUALITY_ORDER = {"NA": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}
OUTLIER_IDS = [524, 1299]


def apply_accepted_preprocessing(df: pd.DataFrame, masvnrtype_path: Path) -> pd.DataFrame:
    """Повторяет предобработку, принятую в baseline_and_preprocessing.ipynb и feature_engineering.ipynb"""
    result = df.copy()
    result[NO_OBJECT_COLUMNS] = result[NO_OBJECT_COLUMNS].fillna("NA")
    result["GarageYrBlt"] = result["GarageYrBlt"].fillna(0)

    raw_masvnrtype = pd.read_csv(masvnrtype_path, keep_default_na=False)["MasVnrType"]
    result["MasVnrType"] = raw_masvnrtype.replace({"NA": np.nan}).values

    result[NO_ZERO_AREA_COLUMNS] = np.log1p(result[NO_ZERO_AREA_COLUMNS])
    result = result.drop(columns=DROPPED_COLUMNS)

    for column in ORDINAL_QUALITY_COLUMNS:
        result[column] = result[column].map(QUALITY_ORDER)

    result["HasPool"] = (result["PoolArea"] > 0).astype(int)
    result["HasMiscFeature"] = (result["MiscFeature"] != "NA").astype(int)
    result = result.drop(columns=["PoolArea", "PoolQC", "MiscFeature", "MiscVal"])

    result["IsRemodeled"] = ((result["YearRemodAdd"] > result["YearBuilt"]) & (result["YearRemodAdd"] > 1950)).astype(
        int
    )
    result["HouseAge"] = result["YrSold"] - result["YearBuilt"]
    return result.drop(columns=["YearBuilt"])


X_base = apply_accepted_preprocessing(X, DATA_DIR / "train.csv")

keep = ~train["Id"].isin(OUTLIER_IDS)
X_base, y_base = X_base[keep].reset_index(drop=True), y[keep].reset_index(drop=True)
log_y_base = np.log1p(y_base)

print(f"признаков: {X_base.shape[1]}, домов: {X_base.shape[0]}")

признаков: 69, домов: 1458


## 2. Схема валидации

KFold с 5 фолдами и перемешиванием, те же разбиения, что в `dnn.ipynb`. Модель обучается на `log1p` цены, RMSE считается на логарифме. Препроцессинг входит в пайплайн каждой модели и подгоняется заново в каждом фолде

`build_preprocessor` теперь умеет отдавать плотный one-hot, он нужен нейросети

In [3]:
def rmse(log_true: np.ndarray, log_pred: np.ndarray) -> float:
    """Среднеквадратичная ошибка между двумя массивами логарифмов цены"""
    return float(np.sqrt(np.mean((log_true - log_pred) ** 2)))


def build_preprocessor(scale: bool, dense: bool = False) -> ColumnTransformer:
    """Медиана для чисел и мода с one-hot для категорий, при dense=True one-hot отдаётся плотным массивом"""
    numeric_steps = [SimpleImputer(strategy="median")] + ([StandardScaler()] if scale else [])
    return ColumnTransformer(
        [
            ("numeric", make_pipeline(*numeric_steps), make_column_selector(dtype_include="number")),
            (
                "categorical",
                make_pipeline(
                    SimpleImputer(strategy="most_frequent"),
                    OneHotEncoder(handle_unknown="ignore", sparse_output=not dense),
                ),
                make_column_selector(dtype_exclude="number"),
            ),
        ]
    )


def cross_validate(model: Pipeline, X: pd.DataFrame, y: pd.Series) -> tuple[float, float]:
    """Считает RMSE на логарифме цены по KFold с 5 фолдами

    Возвращает среднее и стандартное отклонение RMSE по фолдам
    """
    folds = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    log_y = np.log1p(y)
    scores = []
    for fit_idx, valid_idx in folds.split(X):
        fitted = clone(model).fit(X.iloc[fit_idx], log_y.iloc[fit_idx])
        prediction = fitted.predict(X.iloc[valid_idx])
        scores.append(rmse(log_y.iloc[valid_idx].values, prediction))
    return float(np.mean(scores)), float(np.std(scores))


results = []


def evaluate_model(experiment: str, model: Pipeline) -> None:
    """Считает кросс-валидацию модели и записывает результат в results"""
    rmse_value, std_value = cross_validate(model, X_base, y_base)
    results.append({"experiment": experiment, "rmse": rmse_value, "std": std_value})
    print(f"{experiment}: RMSE {rmse_value:.4f}, std {std_value:.4f}")


def show_results() -> pd.DataFrame:
    """Собирает результаты всех экспериментов в таблицу, в порядке добавления"""
    results_df = pd.DataFrame(results)
    experiment_order = results_df["experiment"].drop_duplicates()
    return results_df.set_index("experiment").loc[experiment_order, ["rmse", "std"]].round(4)

## 3. Базовые модели

Берём по одной модели из каждого семейства с лучшими гиперпараметрами из прошлых ноутбуков: Lasso, случайный лес, XGBoost и LightGBM из `optuna_tuning.ipynb`, CatBoost из `classical_models.ipynb`, где Optuna для него не запускалась, и нейросеть из `dnn.ipynb`, один скрытый слой на 64 нейрона с ELU и Dropout 0.3, SGD с lr 1e-2, 400 эпох. Ridge не берём как отдельную модель, он слишком близок к Lasso, он ещё появится как мета-модель в stacking

Нейросеть должна вести себя как обычная sklearn-модель, чтобы ансамбли могли её обучать и вызывать, поэтому оборачиваем её в класс с методами `fit` и `predict`, внутри которого таргет стандартизуется так же, как в `dnn.ipynb`

In [4]:
class NeuralNetRegressor(RegressorMixin, BaseEstimator):
    """Сеть с одним скрытым слоем, ELU и Dropout, обученная SGD на стандартизованном таргете"""

    def __init__(self, hidden_dim=64, dropout=0.3, lr=1e-2, batch_size=32, n_epochs=400, random_state=RANDOM_STATE):
        self.hidden_dim = hidden_dim
        self.dropout = dropout
        self.lr = lr
        self.batch_size = batch_size
        self.n_epochs = n_epochs
        self.random_state = random_state

    def fit(self, X, y):
        X_t = torch.tensor(np.asarray(X, dtype="float32"))
        y = np.asarray(y, dtype="float64")
        self.y_mean_, self.y_std_ = y.mean(), y.std()
        y_t = torch.tensor((y - self.y_mean_) / self.y_std_, dtype=torch.float32).unsqueeze(1)

        torch.manual_seed(self.random_state)
        self.model_ = nn.Sequential(
            nn.Linear(X_t.shape[1], self.hidden_dim), nn.ELU(), nn.Dropout(self.dropout), nn.Linear(self.hidden_dim, 1)
        )
        optimizer = torch.optim.SGD(self.model_.parameters(), lr=self.lr)
        loader = DataLoader(TensorDataset(X_t, y_t), batch_size=self.batch_size, shuffle=True)
        loss_fn = nn.MSELoss()
        for _ in range(self.n_epochs):
            self.model_.train()
            for X_batch, y_batch in loader:
                optimizer.zero_grad()
                loss_fn(self.model_(X_batch), y_batch).backward()
                optimizer.step()
        return self

    def predict(self, X):
        self.model_.eval()
        with torch.no_grad():
            prediction = self.model_(torch.tensor(np.asarray(X, dtype="float32"))).numpy().ravel()
        return prediction * self.y_std_ + self.y_mean_


def build_base_models() -> dict[str, Pipeline]:
    """Базовые модели с лучшими найденными гиперпараметрами, каждая вместе со своим препроцессингом"""
    return {
        "lasso": make_pipeline(
            build_preprocessor(scale=True), Lasso(alpha=0.0005078929047587406, max_iter=20000)
        ),
        "random_forest": make_pipeline(
            build_preprocessor(scale=False),
            RandomForestRegressor(
                n_estimators=118,
                criterion="poisson",
                max_depth=23,
                min_samples_split=4,
                min_samples_leaf=1,
                max_features=0.45997403291690353,
                max_samples=0.9111748662471574,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
        "xgboost": make_pipeline(
            build_preprocessor(scale=False),
            XGBRegressor(
                booster="gbtree",
                n_estimators=990,
                max_depth=3,
                learning_rate=0.05847892915316512,
                min_child_weight=0.637689473011825,
                subsample=0.9029473470866576,
                colsample_bytree=0.4607459901880062,
                gamma=8.285973574448373e-08,
                reg_alpha=4.032976287219069e-07,
                reg_lambda=3.6499272926641786,
                random_state=RANDOM_STATE,
                verbosity=0,
            ),
        ),
        "lightgbm": make_pipeline(
            build_preprocessor(scale=False),
            LGBMRegressor(
                boosting_type="gbdt",
                num_leaves=135,
                max_depth=4,
                learning_rate=0.014191639087470571,
                n_estimators=1310,
                min_child_samples=4,
                subsample=0.8155726506796872,
                subsample_freq=5,
                colsample_bytree=0.43429044707904246,
                reg_alpha=0.0005811844267242152,
                reg_lambda=0.013591235587074845,
                min_split_gain=0.0012512344321506698,
                random_state=RANDOM_STATE,
                verbose=-1,
            ),
        ),
        "catboost": make_pipeline(
            build_preprocessor(scale=False),
            CatBoostRegressor(
                iterations=300,
                learning_rate=0.1,
                depth=4,
                l2_leaf_reg=3,
                random_strength=0,
                bagging_temperature=2,
                rsm=0.7,
                min_data_in_leaf=10,
                random_state=RANDOM_STATE,
                verbose=False,
                thread_count=2,
                allow_writing_files=False,
            ),
        ),
        "dnn": make_pipeline(build_preprocessor(scale=True, dense=True), NeuralNetRegressor()),
    }

Оценим каждую базовую модель по отдельности

In [5]:
for name, model in build_base_models().items():
    evaluate_model(f"1. {name}", model)

1. lasso: RMSE 0.1112, std 0.0054


1. random_forest: RMSE 0.1315, std 0.0068


1. xgboost: RMSE 0.1149, std 0.0065


1. lightgbm: RMSE 0.1136, std 0.0069


1. catboost: RMSE 0.1162, std 0.0076


1. dnn: RMSE 0.1136, std 0.0056


### Выводы: базовые модели

| Модель | RMSE | std |
| --- | --- | --- |
| Lasso | 0.1112 | 0.0054 |
| LightGBM | 0.1136 | 0.0069 |
| Нейросеть | 0.1136 | 0.0056 |
| XGBoost | 0.1149 | 0.0065 |
| CatBoost | 0.1162 | 0.0076 |
| Случайный лес | 0.1315 | 0.0068 |

- Все шесть моделей теперь считаются на одних и тех же фолдах, поэтому таблица сравнима строка со строкой, в отличие от сравнений между прошлыми ноутбуками. Нейросеть с 0.1136 воспроизвела результат из `dnn.ipynb`, то есть обёртка работает так же, как исходный код
- Lasso впереди, 0.1112, LightGBM и нейросеть на одном уровне, 0.1136, следом XGBoost и CatBoost, а случайный лес заметно отстаёт, 0.1315
- Разница между Lasso и следующими четырьмя моделями, от 0.0024 до 0.0050, сопоставима со std по фолдам, около 0.005 и 0.007, поэтому пять сильных моделей близки по качеству, а слабее них только случайный лес
- Гиперпараметры нейросети выбирались по этим же фолдам, а деревьев по другим разбиениям в прошлых ноутбуках, поэтому у нейросети небольшое преимущество отбора

## 4. Усреднение и voting

Пункты 1 и 2 чеклиста. В регрессии нет голосования по классам, поэтому voting понимаем как взвешенное среднее предсказаний, а усреднение как обычное среднее с равными весами. Вес модели во взвешенном среднем обратно пропорционален квадрату её RMSE из таблицы выше, чем модель точнее, тем больше её вклад. Эти RMSE посчитаны на тех же фолдах, поэтому веса слегка подстроены под них

Каждую схему считаем дважды, на всех шести моделях и на пяти без случайного леса, который заметно слабее остальных

In [6]:
from sklearn.ensemble import VotingRegressor

single_rmse = pd.DataFrame(results).set_index("experiment")["rmse"]
base_names = list(build_base_models())
inverse_square_weights = {name: 1 / single_rmse[f"1. {name}"] ** 2 for name in base_names}


def build_voting(names: list[str], weights: dict[str, float] | None = None) -> VotingRegressor:
    """Собирает VotingRegressor из выбранных базовых моделей, weights по умолчанию равные"""
    models = build_base_models()
    return VotingRegressor(
        [(name, models[name]) for name in names],
        weights=[weights[name] for name in names] if weights else None,
    )


model_sets = {
    "6 моделей": base_names,
    "5 моделей без леса": [name for name in base_names if name != "random_forest"],
}

for label, names in model_sets.items():
    evaluate_model(f"2. усреднение, {label}", build_voting(names))
    evaluate_model(f"3. voting с весами, {label}", build_voting(names, inverse_square_weights))

2. усреднение, 6 моделей: RMSE 0.1098, std 0.0064


3. voting с весами, 6 моделей: RMSE 0.1094, std 0.0064


2. усреднение, 5 моделей без леса: RMSE 0.1087, std 0.0064


3. voting с весами, 5 моделей без леса: RMSE 0.1087, std 0.0064


### Выводы: усреднение и voting

Лучшая одиночная модель, Lasso, дала 0.1112 со std 0.0054

| Схема | Модели | RMSE | std |
| --- | --- | --- | --- |
| Усреднение | 5 без леса | 0.1087 | 0.0064 |
| Voting с весами | 5 без леса | 0.1087 | 0.0064 |
| Voting с весами | 6 | 0.1094 | 0.0064 |
| Усреднение | 6 | 0.1098 | 0.0064 |

- Все четыре ансамбля лучше любой одиночной модели, включая Lasso, и лучше всего, что было в прошлых ноутбуках, где лучшим было 0.1110 у Lasso
- Случайный лес ансамблю мешает: без него усреднение улучшается с 0.1098 до 0.1087, он слабее остальных, 0.1315 против 0.1112 и 0.1162, а в простом среднем его вклад такой же, как у сильных моделей
- Веса обратно пропорциональные квадрату RMSE почти ничего не меняют: на шести моделях выигрыш всего 0.0004, а на пяти без леса результат совпал до четвёртого знака. Пять оставшихся моделей близки по качеству, поэтому их веса почти равны, а вес леса и так был лишь немного ниже
- Выигрыш у Lasso, 0.0025, меньше std по фолдам, 0.0054 и 0.0064. Парную разницу по фолдам мы не считали, поэтому надёжность этого выигрыша по одним этим числам оценить нельзя, но ансамбль обходит все пять одиночных моделей одновременно, а не одну из них
- Лучший результат ансамблей пока 0.1087, это усреднение или voting на пяти моделях без леса

## 5. Stacking

Пункт 3 чеклиста: мета-модель учится по предсказаниям базовых моделей. В качестве мета-модели берём обычную линейную регрессию без регуляризации и Ridge с alpha, подобранным внутри `RidgeCV`. Внутренняя кросс-валидация, на которой строятся предсказания для мета-модели, использует 3 фолда вместо 5 по умолчанию, иначе каждый базовый бустинг и нейросеть обучаются слишком много раз

Наборы моделей те же, что в voting, шесть и пять без случайного леса

In [6]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression, RidgeCV


def build_stacking(names: list[str], final_estimator) -> StackingRegressor:
    """Собирает StackingRegressor из выбранных базовых моделей с перемешанной внутренней кросс-валидацией"""
    models = build_base_models()
    return StackingRegressor(
        [(name, models[name]) for name in names],
        final_estimator=final_estimator,
        cv=KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    )


meta_models = {
    "LinearRegression": LinearRegression,
    "Ridge": lambda: RidgeCV(alphas=np.logspace(-3, 3, 13)),
}

for label, names in model_sets.items():
    for meta_name, build_meta in meta_models.items():
        evaluate_model(f"4. stacking {meta_name}, {label}", build_stacking(names, build_meta()))

4. stacking LinearRegression, 6 моделей: RMSE 0.1092, std 0.0062


4. stacking Ridge, 6 моделей: RMSE 0.1091, std 0.0063


4. stacking LinearRegression, 5 моделей без леса: RMSE 0.1090, std 0.0062


4. stacking Ridge, 5 моделей без леса: RMSE 0.1089, std 0.0063


### Выводы: stacking

| Мета-модель | Модели | RMSE | std |
| --- | --- | --- | --- |
| Ridge | 5 без леса | 0.1089 | 0.0063 |
| LinearRegression | 5 без леса | 0.1090 | 0.0062 |
| Ridge | 6 | 0.1091 | 0.0063 |
| LinearRegression | 6 | 0.1092 | 0.0062 |

- Stacking не обошёл простое усреднение: лучший stacking, 0.1089, чуть хуже усреднения и voting на пяти моделях без леса, 0.1087, а разница между всеми ансамблями, от 0.1087 до 0.1098, намного меньше std по фолдам, около 0.006
- Линейная регрессия без регуляризации и Ridge дают практически одно и то же, разница 0.0001. Мета-модель работает всего с пятью или шестью признаками на 1166 обучающих домах в фолде, и регуляризация ей ничего не добавляет
- Случайный лес в stacking мешает гораздо меньше, чем в простом среднем: разница между шестью и пятью моделями 0.0002 против 0.0011 у усреднения. Веса мета-модели мы не выводили, поэтому причину этого не проверяли
- Итог по ансамблям в целом: любой из них лучше лучшей одиночной модели, Lasso с 0.1112, и все они между собой неразличимы по качеству

## 6. Ансамбли без нейросети

Чтобы понять, что нейросеть добавляет ансамблю, повторяем все схемы на четырёх моделях без случайного леса и без нейросети: Lasso, XGBoost, LightGBM и CatBoost. Сравнивать нужно с пятью моделями без леса, где нейросеть есть

In [8]:
no_dnn_names = [name for name in base_names if name not in ("random_forest", "dnn")]

evaluate_model("5. усреднение, 4 модели без леса и нейросети", build_voting(no_dnn_names))
evaluate_model(
    "5. voting с весами, 4 модели без леса и нейросети", build_voting(no_dnn_names, inverse_square_weights)
)
for meta_name, build_meta in meta_models.items():
    evaluate_model(
        f"5. stacking {meta_name}, 4 модели без леса и нейросети", build_stacking(no_dnn_names, build_meta())
    )

5. усреднение, 4 модели без леса и нейросети: RMSE 0.1095, std 0.0066


5. voting с весами, 4 модели без леса и нейросети: RMSE 0.1094, std 0.0066


5. stacking LinearRegression, 4 модели без леса и нейросети: RMSE 0.1085, std 0.0059


5. stacking Ridge, 4 модели без леса и нейросети: RMSE 0.1084, std 0.0060


### Выводы: вклад нейросети

| Схема | 4 модели без нейросети | 5 моделей с нейросетью | Разница |
| --- | --- | --- | --- |
| Усреднение | 0.1095 | 0.1087 | нейросеть лучше на 0.0008 |
| Voting с весами | 0.1094 | 0.1087 | нейросеть лучше на 0.0007 |
| Stacking LinearRegression | 0.1085 | 0.1090 | нейросеть хуже на 0.0005 |
| Stacking Ridge | 0.1084 | 0.1089 | нейросеть хуже на 0.0005 |

- Эффект нейросети зависит от схемы: в простом среднем и в voting она помогает, в stacking мешает. Во всех четырёх случаях разница от 0.0005 до 0.0008, что на порядок меньше std по фолдам, 0.006, поэтому надёжного вклада нейросети в ансамбль этот эксперимент не показывает, ни положительного, ни отрицательного
- Лучший результат всего ноутбука теперь у stacking без нейросети, Ridge с 0.1084 и std 0.0060, но он отличается от усреднения с нейросетью, 0.1087, на 0.0003, то есть неотличим от него
- Нейросеть сама по себе на уровне LightGBM, 0.1136, но в ансамбле её вклад лежит в пределах шума. Если ансамбль нужно упростить, её можно убрать без заметной потери качества

## Итоги ноутбука

Прошли все пункты чеклиста для ансамблей: усреднение, voting и stacking через линейную регрессию и Ridge. Базовые модели взяты с лучшими гиперпараметрами из прошлых ноутбуков, все результаты посчитаны на одних и тех же фолдах KFold с 5 разбиениями, поэтому таблица ниже сравнима строка со строкой

Все результаты, от лучшего к худшему:

| Модель | Тип | RMSE | std |
| --- | --- | --- | --- |
| Stacking Ridge, 4 модели без леса и нейросети | ансамбль | 0.1084 | 0.0060 |
| Stacking LinearRegression, 4 модели без леса и нейросети | ансамбль | 0.1085 | 0.0059 |
| Усреднение, 5 моделей без леса | ансамбль | 0.1087 | 0.0064 |
| Voting с весами, 5 моделей без леса | ансамбль | 0.1087 | 0.0064 |
| Stacking Ridge, 5 моделей без леса | ансамбль | 0.1089 | 0.0063 |
| Stacking LinearRegression, 5 моделей без леса | ансамбль | 0.1090 | 0.0062 |
| Stacking Ridge, 6 моделей | ансамбль | 0.1091 | 0.0063 |
| Stacking LinearRegression, 6 моделей | ансамбль | 0.1092 | 0.0062 |
| Voting с весами, 6 моделей | ансамбль | 0.1094 | 0.0064 |
| Voting с весами, 4 модели без леса и нейросети | ансамбль | 0.1094 | 0.0066 |
| Усреднение, 4 модели без леса и нейросети | ансамбль | 0.1095 | 0.0066 |
| Усреднение, 6 моделей | ансамбль | 0.1098 | 0.0064 |
| Lasso | одиночная | 0.1112 | 0.0054 |
| LightGBM | одиночная | 0.1136 | 0.0069 |
| Нейросеть | одиночная | 0.1136 | 0.0056 |
| XGBoost | одиночная | 0.1149 | 0.0065 |
| CatBoost | одиночная | 0.1162 | 0.0076 |
| Случайный лес | одиночная | 0.1315 | 0.0068 |

Что показали эксперименты:

- **Любой ансамбль лучше любой одиночной модели.** Худший ансамбль, усреднение шести моделей с 0.1098, всё равно лучше лучшей одиночной модели, Lasso с 0.1112. Выигрыш лежит от 0.0014 до 0.0028, что меньше std по фолдам, около 0.006, парную разницу по фолдам мы не считали, но ансамбли обходят Lasso все двенадцать раз из двенадцати
- **Выбор схемы ансамбля почти не влияет.** Весь разброс между двенадцатью ансамблями от 0.1084 до 0.1098, то есть 0.0014, что в четыре раза меньше std, поэтому усреднение, voting и stacking с линейной регрессией или Ridge неразличимы по качеству. Ridge и линейная регрессия без регуляризации в роли мета-модели дают одно и то же
- **Случайный лес ансамблю мешает в простом среднем**, 0.1098 с ним против 0.1087 без него, а в stacking почти нет, разница 0.0002. **Нейросеть** улучшает усреднение и voting на 0.0008 и 0.0007 и ухудшает stacking на 0.0005, надёжного вклада в ансамбль у неё нет
- **Оценка немного оптимистична.** Гиперпараметры моделей подбирались на тех же данных, для нейросети по этим же фолдам, а веса в voting посчитаны по RMSE с этих же фолдов, поэтому реальный выигрыш на новых данных, вероятно, скромнее

Сравнение с прошлыми ноутбуками: лучший результат одиночной модели там был 0.1110 у Lasso через Optuna, а лучшие ансамбли здесь дают 0.1084 и 0.1087. Схемы валидации отличаются, там RepeatedKFold с тремя повторами, здесь один KFold, поэтому сравнение приблизительное, но порядок величины выигрыша, около 0.002 до 0.003, совпадает с внутренним сравнением в таблице выше

Так как по качеству все ансамбли без случайного леса равноценны, выбор для финального сабмита можно делать по простоте и стоимости обучения, а не по RMSE

## Результаты на лидерборде

Семь моделей и ансамблей из этого ноутбука обучены на всём трейне и отправлены на Kaggle, файлы `<имя>_submission.csv` лежат в папке `submissions`. Публичные оценки забираем прямо из Kaggle CLI, для запуска ячейки он должен быть настроен, а оценка на кросс-валидации берётся из таблицы результатов выше

Метрика соревнования та же, RMSE на логарифме цены, поэтому оценки на лидерборде и на кросс-валидации можно сравнивать напрямую

In [6]:
import subprocess
import sys
from io import StringIO

COMPETITION = "house-prices-advanced-regression-techniques"

cv_experiment_by_file = {
    "lasso": "1. lasso",
    "xgboost": "1. xgboost",
    "lightgbm": "1. lightgbm",
    "catboost": "1. catboost",
    "dnn": "1. dnn",
    "averaging_5_models": "2. усреднение, 5 моделей без леса",
    "stacking_ridge_4_models": "5. stacking Ridge, 4 модели без леса и нейросети",
}

kaggle_output = subprocess.run(
    [sys.executable, "-m", "kaggle.cli", "competitions", "submissions", "-c", COMPETITION, "-v"],
    capture_output=True,
    text=True,
    check=True,
).stdout
submissions = pd.read_csv(StringIO(kaggle_output)).sort_values("date").drop_duplicates("fileName", keep="last")
public_score = submissions.set_index("fileName")["publicScore"]
cv_rmse = pd.DataFrame(results).drop_duplicates("experiment").set_index("experiment")["rmse"]

leaderboard = pd.DataFrame(
    {
        "CV RMSE": {name: cv_rmse[experiment] for name, experiment in cv_experiment_by_file.items()},
        "Лидерборд": {name: public_score[f"{name}_submission.csv"] for name in cv_experiment_by_file},
    }
)
leaderboard["Разница"] = leaderboard["Лидерборд"] - leaderboard["CV RMSE"]
leaderboard.sort_values("Лидерборд").round(4)

,CV RMSE,Лидерборд,Разница
averaging_5_models,0.1087,0.1234,0.0147
stacking_ridge_4_models,0.1084,0.1242,0.0158
dnn,0.1136,0.1263,0.0127
lightgbm,0.1136,0.1273,0.0137
lasso,0.1112,0.1276,0.0164
xgboost,0.1149,0.1282,0.0133
catboost,0.1162,0.1288,0.0126


### Выводы: лидерборд

- **Ансамбли на первых двух местах.** Усреднение пяти моделей даёт 0.1234, stacking Ridge на четырёх 0.1242, а лучшая одиночная модель, нейросеть, 0.1263. Усреднение обходит её на 0.0029, то есть вывод из кросс-валидации о том, что ансамбль лучше одиночных моделей, подтвердился на тестовых данных
- **Порядок двух ансамблей не совпал.** На кросс-валидации stacking чуть лучше, 0.1084 против 0.1087, на лидерборде усреднение лучше, 0.1234 против 0.1242. Разница в обоих случаях меньше 0.001, поэтому по этим данным между ними выбрать нельзя, усреднение проще и не требует внутренней кросс-валидации
- **Лидерборд хуже кросс-валидации у всех семи, на 0.0126 до 0.0164.** Сдвиг в одну сторону и примерно одного размера, поэтому общая картина сохраняется, но абсолютные числа с кросс-валидации выглядят оптимистичнее реальных. Причину сдвига мы не исследовали
- **Порядок одиночных моделей на кросс-валидации не воспроизвёлся.** Lasso была лучшей одиночной моделью, 0.1112, а на лидерборде стала пятой из семи, 0.1276, с самым большим сдвигом, 0.0164. Нейросеть и LightGBM, которые на кросс-валидации были на равных, 0.1136, оказались первой и второй среди одиночных, 0.1263 и 0.1273. Все пять одиночных моделей на лидерборде лежат в диапазоне 0.0025, от 0.1263 до 0.1288, а на кросс-валидации в диапазоне 0.0050, так что различия между ними невелики и на лидерборде
- **Дом Id 2550 сам по себе порядок не объясняет.** Lasso и нейросеть предсказали для него больше 1,1 млн, бустинги около 0,7 млн, но нейросеть на лидерборде лучшая из одиночных, а Lasso пятая, то есть высокое предсказание не наказало обе модели одинаково. Настоящую цену этого дома мы не знаем
- **Кандидат для финального сабмита.** Усреднение пяти моделей, лучшее на лидерборде и почти лучшее на кросс-валидации